# Notebook para Análise de Dados com PokeAPI

- **Objetivo**.

Nesta etapa, o(a) candidato(a) deverá consumir dados diretamente da API pública da PokeAPI e
realizar análises utilizando Apache Spark.O objetivo é avaliar habilidades de:

    - Ingestão de dados a partir de API REST
    - Tratamento de paginação e dados aninhados (JSON)
    - Modelagem relacional
    - Manipulação e agregação de dados com Spark
    - Clareza na organização e documentação do código

In [ ]:
import sys
import asyncio
import aiohttp
from aiohttp import ClientSession
from typing import List, Dict, Any
import json
import logging
from tenacity import retry, wait_exponential, stop_after_attempt

logging.basicConfig(level=logging.INFO)

BASE_URL = "https://pokeapi.co/api/v2/pokemon/"
CONCURRENT_REQUESTS_LIMIT = 10

## Etapa 1 — Data Extraction

Consumir o endpoint /pokemon para obter a listagem completa de todos os pokémons.

1. Para cada url retornada, realizar uma nova requisição e extrair exclusivamente os seguintes
campos e transformá los em novas tabelas:
types, stats, abilities

In [ ]:
class PokemonFetchData:
    def __init__(self, base_url: str, concurrent_requests_limit: int = 10):
        self.base_url = base_url
        self.semaphore = asyncio.Semaphore(concurrent_requests_limit)

    @retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
    async def _fetch_url_json(self,
                            session: aiohttp.ClientSession, 
                            url: str,
                            semaphore: asyncio.Semaphore
                            ) -> Dict[str, Any]:
        """ Internal Method to fetch JSON data from a given URL using aiohttp with retry logic.
        Args:
            session (aiohttp.ClientSession): The aiohttp session to use for the request.
            url (str): The URL to fetch data from.
            semaphore (asyncio.Semaphore): Semaphore to limit concurrent requests.
        Returns:
            Dict[str, Any]: The JSON data fetched from the URL, or an empty dictionary in case of an error.
        """
        async with semaphore:
            try:
                async with session.get(url) as response:
                    response.raise_for_status()
                    return await response.json()
            except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                logging.error(f"Error fetching {url}: {e}")
                return {}


    async def _fetch_all_pokemons_base_urls(self) -> List[Dict[str, Any]]:
        """Internal Method to fetch all pokemon URLs from the PokeAPI.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the base URLs of all pokemons.
        """
        timeout = aiohttp.ClientTimeout(total=30)
        all_pokemon_data = []
        try:
            async with aiohttp.ClientSession(timeout=timeout) as session:
                data = await self._fetch_url_json(session, self.base_url, self.semaphore)
                all_pokemon_data.extend(data.get("results", []))

                while data.get("next"):
                    next_url = data["next"]
                    data = await self._fetch_url_json(session, next_url, self.semaphore)
                    all_pokemon_data.extend(data.get("results", []))
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            logging.error(f"Error fetching all pokemons: {e}")
        return all_pokemon_data


    async def _fetch_pokemon_raw_data(self,
                                     pokemon_urls_list:List[Dict[str,Any]]
                                     ) -> Dict[str, Any]:
        """ Internal Method to fetch each pokemon types, stats, and abilities from the PokeAPI using the list of pokemon URLs.
        
        Args:
            pokemon_urls_list (List[Dict[str, Any]]): List of dictionaries containing pokemon names and their corresponding URLs.
        Returns:
            Dict[str, Any]: A list of dictionaries containing pokemon names and their corresponding types, stats, abilities, height, weight, and base_experience data.
        """
        async with self.semaphore:
            async with aiohttp.ClientSession() as session:
                try:
                    data = [self._fetch_url_json(session, pokemon["url"], self.semaphore) for pokemon in pokemon_urls_list]
                    pokemon_fetch_data = await asyncio.gather(*data)
                except Exception as e:
                    logging.error(f"Error fetching pokemon raw data: {e}")

                pokemons_raw_data = [ 
                    {"name": pokemon["name"],
                    "pokemon_id": payload["id"],
                    "height": payload["height"],
                    "weight": payload["weight"],
                    "base_experience": payload["base_experience"],
                    "types": payload["types"], 
                    "stats": payload["stats"], 
                    "abilities": payload["abilities"]} 
                    for pokemon, payload in zip(pokemon_urls_list, pokemon_fetch_data) if payload
                    ]
                return pokemons_raw_data


    def create_tables(self, pokemons_full_raw_data: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
        """Function to create tables for each pokemon with their corresponding stats.
        
        Args:
            pokemons_full_raw_data (List[Dict[str, Any]]): A list of dictionaries containing pokemon names and their corresponding status data.
        Returns:
            Dict[str, List[Dict[str, Any]]]: A dictionary containing tables for each pokemon with their corresponding stats.
        """
        pokemon_table = []
        pokemon_type_table = []
        pokemon_stats_table = []
        pokemon_ability_table = []

        

    async def fetch_pokemons(self, pokemons_raw_data_address: str):
        """Function to fetch all pokemons from the PokeAPI and save their raw data to a JSON file.

        Args:
            pokemons_raw_data_address (str): The file path to save the raw pokemon data.
        """
        pokemons_base_data = await self._fetch_all_pokemons_base_urls()

        pokemons_raw_data = await self._fetch_pokemon_raw_data(pokemons_base_data)

        with open(pokemons_raw_data_address, "w", encoding="utf-8") as f:
            json.dump(pokemons_raw_data, f, indent=2, ensure_ascii=False)



In [ ]:
pokemon_object = PokemonFetchData(base_url=BASE_URL, concurrent_requests_limit=CONCURRENT_REQUESTS_LIMIT)
pokemons_raw_data_address = "data/raw/pokemons_raw_data.json"
await pokemon_object.fetch_pokemons(pokemons_raw_data_address)



In [ ]:
@retry(wait=wait_exponential(min=1, max=10), stop=stop_after_attempt(3))
async def fetch_url_json(session: aiohttp.ClientSession, 
                        url: str,
                        semaphore: asyncio.Semaphore
                        ) -> Dict[str, Any]:
    """Function to fetch JSON data from a given URL using aiohttp with retry logic.
    Args:
        session (aiohttp.ClientSession): The aiohttp session to use for the request.
        url (str): The URL to fetch data from.
        semaphore (asyncio.Semaphore): Semaphore to limit concurrent requests.
    Returns:
        Dict[str, Any]: The JSON data fetched from the URL, or an empty dictionary in case of an error.
    """
    async with semaphore:
        try:
            async with session.get(url) as response:
                return await response.json()
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            logging.error(f"Error fetching {url}: {e}")
            return {}

In [ ]:
async def fetch_all_pokemons_base_urls() -> List[Dict[str, Any]]:
    """Function to fetch all pokemon URLs from the PokeAPI.
    Returns:
        List[Dict[str, Any]]: A list of dictionaries containing the base URLs of all pokemons.
    """
    timeout = aiohttp.ClientTimeout(total=30)
    all_pokemon_data = []
    try:
        async with aiohttp.ClientSession(timeout=timeout) as session:
            data = await fetch_url_json(session, BASE_URL, SEMAPHORE)
            all_pokemon_data.extend(data["results"])

            while data.get("next"):
                next_url = data["next"]
                data = await fetch_url_json(session, next_url, SEMAPHORE)
                all_pokemon_data.extend(data["results"])
    except (aiohttp.ClientError, asyncio.TimeoutError) as e:
        logging.error(f"Error fetching all pokemons: {e}")
    return all_pokemon_data

pokemon_urls_list = await fetch_all_pokemons_base_urls()

with open("data/pokemon_urls_list.json", "w", encoding="utf-8") as f:
    json.dump(pokemon_urls_list, f, indent=2, ensure_ascii=False)

In [ ]:
async def fetch_pokemon_raw_data(pokemon_urls_list:List[Dict[str,Any]]) -> Dict[str, Any]:
    """ Function to fetch each pokemon types, stats, and abilities from the PokeAPI using the list of pokemon URLs.
    
    Args:
        pokemon_urls_list (List[Dict[str, Any]]): List of dictionaries containing pokemon names and their corresponding URLs.
    Returns:
        Dict[str, Any]: A list of dictionaries containing pokemon names and their corresponding types, stats, and abilities data.
    """
    async with SEMAPHORE:
        async with aiohttp.ClientSession() as session:
            try:
                data = [fetch_url_json(session, pokemon["url"], SEMAPHORE) for pokemon in pokemon_urls_list]
                pokemon_fetch_data = await asyncio.gather(*data)
            except Exception as e:
                logging.error(f"Error fetching pokemon raw data: {e}")

            pokemons_raw_data = [{"name": pokemon["name"],
                                  "pokemon_id": payload["id"],
                                  "height": payload["height"],
                                  "weight": payload["weight"],
                                  "base_experience": payload["base_experience"],
                                  "types": payload["types"], 
                                  "stats": payload["stats"], 
                                  "abilities": payload["abilities"]} 
                                  for pokemon, payload in zip(pokemon_urls_list, pokemon_fetch_data) if payload]
            return pokemons_raw_data

pokemons_raw_data = await fetch_pokemon_raw_data(pokemon_urls_list)

with open("data/pokemons_raw_data.json", "w", encoding="utf-8") as f:
    json.dump(pokemons_raw_data, f, indent=2, ensure_ascii=False)

In [ ]:
def create_pokemon_tables(pokemons_full_raw_data: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    """Function to create tables for each pokemon with their corresponding stats.
    
    Args:
        pokemons_full_raw_data (List[Dict[str, Any]]): A list of dictionaries containing pokemon names and their corresponding status data.
    Returns:
        Dict[str, List[Dict[str, Any]]]: A dictionary containing tables for each pokemon with their corresponding stats.
    """
    

## Etapa 2 — Data Modeling

Ao final da extração, os dados deverão estar organizados de acordo com as tabelas apresentadas no
tópico Dicionario de Dados na seção Anexos no final desse documento.

## Etapa 3 — Data Analysis using Spark

Após a construção das tabelas, o(a) candidato(a) deverá responder às seguintes perguntas utilizando
Apache Spark:
Caso você não tenha um ambiente Spark, tente ver o Databricks Community Edition.